# Connecting to Databricks from RStudio

You can connect to Databricks from RStudio in a similar way to how we currently use SQL Server from RStudio.


# Before you start

Before you do this you will need,
- Access to the Databricks platofrm
- Access to a SQL warehouse **or** a Personal Cluster on Databricks
- R and RStudio downloaded and installed (with the `odbc`, `DBI`, and `usethis` packages installed)

If you have not got acess to Databricks or an SQL warehouse / Personal Cluster, review the Databricks fundementals course materials.

## SQL warehouse or Personal Cluster

When connecting to Databricks, you can use either an SQL warehouse or Personal Cluser. Each connection method offers some differences in functionality.

**We recommend you use an SQL Warehouse.** This is because they are faster to start and optimised for SQL. This makes them ideal if you already have existing RAP pipelines set up using SQL scripts in a Git repo.

However, Personal Clusters are more flexible, supporting multiple programming languages (R, Python, and Scala,) and being able to access volumes within the Unity Catalog.

If you only need to access tables in the Unity Catalog and only need to use SQL, use a SQL Warehouse, otherwise, use a Personal Cluster.

# The Process

To establish the connection, you will need to do three things,
1) Install an ODBC driver onto your laptop
2) Modify your .Renviron file
3) Write code to connect to Databricks from RStudio

Lets go through each step in more detail

## Install the Simba Spark ODBC driver

This is needed to enable the connection between your laptop and Databricks.

You can install this from the Software Centre on your laptop (which you can find by clicking the window icon on the bottom left of your screen and seaching for it)

On the **Applications** tab on the left hand menu, search for `Simba Spark ODBC Driver 64-bit` and click install.

![locating_ODBC_driver.png](./../../../images/ConnectingToDatabricks/locating_ODBC_driver.png "locating_ODBC_driver.png")

## Modify your .Renviron file

This is needed to establish a connection between RStudio and Databricks, by creating and modifying environment variablies in your .Renviron file.

To open this file you will need to run `usethis::edit_r_environ()` in the console.

You will need to find the following information to add to this file;

1) **DATABRICKS_HOST**

    This is the instance of Databricks you want to connect to. When in Databricks, copy the "https://" path up to and including "azuredatabricks.net". Ignore anything after that.

2) **DATABRICKS_SQL_PATH / DATABRICKS_CLUSTER_PATH**
    
    You should pick the one you have access to from the rerequisities for this session (a SQL warehouse or Personal Cluster on Databricks).

    For the SQL warehouse, the path can be found by clicking on **SQL Warehouses** under the "SQL" section of the left hand menu on Databricks. On this page, click on the warehouse name that you would like to get the path for. On the **Connection details** tab the SQL warehouse path is named "HTTP path", and should start with something similar to “/sql/1.0/warehouses/".

    For the Personal Cluster, the path can be found by clicking on **Compute** in the left hand menu on Databricks. On this page, click on the cluster name that you would like to get the path for. On the **Configuration** tab, under Advanced options, select JDBC / ODBC. This will reveal the path under ""HTTP path", and should start with someting similar to "sql/protocolv1/o/".

3) **DATABRICKS_TOKEN**

    This is your personal access token, a security measure that acts as an identifier to let Databricks know who you are when accessing information.

    **They are only usable for a limited number of days so will need periodically renewing**

    You can find this by clicking on your icon in the top right corner of databricks. In **Settings** go to the **Developer** section in the left hand menu. Here there will be a section for Access tokens, click "Manage".

    ![locating_personal_access_tokens.png](./../../../images/ConnectingToDatabricks/locating_personal_access_tokens.png "locating_personal_access_tokens.png")

    Here, click to "Generate new token".
    
    You will need to; name the token someting sensible so you can itentify it, set the lifetime (to a reasonable period of time 90 days is industry standard), and set the scope to Other APIs, with **all-apis** selected from the drop down menu below.

    ![specifying_token_requirements.png](./../../../images/ConnectingToDatabricks/specifying_token_requirements.png "specifying_token_requirements.png")
    
    Then generate it!
    
    Take note of the Lifetime you have set, this will tell you how long you will have this token before you make another.
    
    **Copy the Databricks access token it generates before you leave this page, you will not be able to come back to see this again through Databricks**

Your .Renviron file should now look like this, with the information you found above copied into the quotation marks,

DATABRICKS_HOST="databricks-host-url"

DATABRICKS_SQL_PATH="sql-warehouse-path" **OR** DATABRICKS_CLUSTER_PATH="personal-cluster-path"

DATABRICKS_TOKEN="personal-access-token"

## Write code to pull data from Databricks into RStudio

Now you have your ODBC driver installed, and .Renviron variables set, you can start writing codes to explore data on Databricks.

You will use the `odbc` and `DBI` package for this.

To set up the connection, you will need to add the following code to the start of your script,

```
library(odbc)
library(DBI)

con <- DBI::dbConnect(
  odbc::databricks(),
  driver = "Databricks ODBC Driver",
  httpPath = Sys.getenv("DATABRICKS_SQL_PATH")
)
```
**NOTE**: If you are using a Personal Cluster, you will need to replace DATABRICKS_SQL_PATH with DATABRICKS_CLUSTER_PATH.


In [0]:
%python
displayHTML("""
<style>
  .quiz-container { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 700px; padding: 1rem; }
  .quiz-title { font-size: 1.5rem; font-weight: 600; margin-bottom: 1.5rem; color: #1a1a1a; }
  .question { margin-bottom: 1.5rem; padding: 1rem; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; }
  .question h3 { margin: 0 0 0.75rem 0; font-size: 1rem; color: #333; }
  .options label { display: block; padding: 0.5rem 0.75rem; margin: 0.25rem 0; border-radius: 4px; cursor: pointer; transition: background 0.2s; }
  .options label:hover { background: #e8f0fe; }
  .options input[type='radio'] { margin-right: 0.5rem; }
  .feedback { margin-top: 0.5rem; padding: 0.5rem 0.75rem; border-radius: 4px; font-weight: 500; display: none; }
  .feedback.correct { background: #d4edda; color: #155724; display: block; }
  .feedback.incorrect { background: #f8d7da; color: #721c24; display: block; }
  .submit-btn { margin-top: 1.5rem; padding: 0.6rem 1.5rem; background: #0066cc; color: white; border: none; border-radius: 4px; font-size: 1rem; cursor: pointer; }
  .submit-btn:hover { background: #0052a3; }
  .score { margin-top: 1rem; font-size: 1.1rem; font-weight: 600; color: #333; }
</style>

<div class="quiz-container">
  <div class="quiz-title">&#x2705; Knowledge Check: Connecting to Databricks from RStudio</div>

  <div class="question" id="q1">
    <h3>1. Which is a benefit of connecting to Databricks via an SQL warehouse over a Personal Cluster?</h3>
    <div class="options">
      <label><input type="radio" name="q1" value="a"> SQL Warehouses are faster to start and optimised for SQL</label>
      <label><input type="radio" name="q1" value="b"> SQL Warehouses are more flexible and support R, Python and Scala</label>
      <label><input type="radio" name="q1" value="c"> SQL Warehouses can access volumes within the Unity Catalog</label>
      <label><input type="radio" name="q1" value="d"> There is no practical difference; they provide identical functionality</label>
    </div>
    <div class="feedback" id="fb1"></div>
  </div>

  <div class="question" id="q2">
    <h3>2. Where can you install the Simba Spark ODBC Driver?</h3>
    <div class="options">
      <label><input type="radio" name="q2" value="a"> From CRAN Package Repository</label>
      <label><input type="radio" name="q2" value="b"> From a Windows Update</label>
      <label><input type="radio" name="q2" value="c"> From the Software Centre on your laptop</label>
      <label><input type="radio" name="q2" value="d"> From Databricks Workspace</label>
    </div>
    <div class="feedback" id="fb2"></div>
  </div>

  <div class="question" id="q3">
    <h3>3. What should the DATABRICKS_HOST environment variable contain?</h3>
    <div class="options">
      <label><input type="radio" name="q3" value="a"> Your email address</label>
      <label><input type="radio" name="q3" value="b"> The name of your SQL Warehouse</label>
      <label><input type="radio" name="q3" value="c"> The Databricks URL up to and including azuredatabricks.net</label>
      <label><input type="radio" name="q3" value="d"> Your personal access token</label>
    </div>
    <div class="feedback" id="fb3"></div>
  </div>

  <div class="question" id="q4">
    <h3>4. If you are connecting through a SQL Warehouse, which environment variable should contain the HTTP Path?</h3>
    <div class="options">
      <label><input type="radio" name="q4" value="a"> DATABRICKS_HOST</label>
      <label><input type="radio" name="q4" value="b"> DATABRICKS_SQL_PATH</label>
      <label><input type="radio" name="q4" value="c"> DATABRICKS_CONNECTION</label>
      <label><input type="radio" name="q4" value="d"> DATABRICKS_TOKEN</label>
    </div>
    <div class="feedback" id="fb4"></div>
  </div>

  <div class="question" id="q5">
    <h3>5. Which statement about Databricks Personal Access Tokens is correct?</h3>
    <div class="options">
      <label><input type="radio" name="q5" value="a"> They only are required by one member in a team</label>
      <label><input type="radio" name="q5" value="b"> They can be viewed again later in Databricks</label>
      <label><input type="radio" name="q5" value="c"> They are only needed when using Personal Clusters</label>
      <label><input type="radio" name="q5" value="d"> They expire after a set period</label>
    </div>
    <div class="feedback" id="fb5"></div>
  </div>

  <button class="submit-btn" onclick="checkAnswers()">Check Answers</button>
  <div class="score" id="score"></div>
</div>

<script>
  const answers = { q1: 'a', q2: 'c', q3: 'c', q4: 'b', q5: 'd' };
  const explanations = {
    q1: "SQL Warehouses are designed specifically for SQL workloads and are quick to start, meaning they are recommended if you are migrating an existing RAP pipeline set up using SQL scripts.",
    q2: "You install Simba Spark ODBC Driver from the Software Centre on your laptop (which you can find by clicking the window icon on the bottom left of your screen and seaching for it).",
    q3: "The DATABRICKS_HOST environment variable is the instance of Databricks you want to connect to, and should contain the Databricks URL from 'https://' up to and including 'azuredatabricks.net'",
    q4: "You can find the HTTP Path for your SQL Warehouse in the Databricks UI, by clicking on your SQL Warehouse, this needs to be assigned to DATABRICKS_SQL_PATH in your .renviron file.",
    q5: "Databricks Personal Access Tokens are personal to you, and will expire after a set time frame, you also won't be able to access it again once its been created, so make sure to copy it straight away."
  };

  function checkAnswers() {
    let score = 0;
    for (let q in answers) {
      const selected = document.querySelector(`input[name="${q}"]:checked`);
      const fb = document.getElementById('fb' + q.charAt(1));
      if (selected && selected.value === answers[q]) {
        score++;
        fb.className = 'feedback correct';
        fb.innerHTML = '\u2705 Correct! ' + explanations[q];
      } else if (selected) {
        fb.className = 'feedback incorrect';
        fb.innerHTML = '\u274c Incorrect. ' + explanations[q];
      } else {
        fb.className = 'feedback incorrect';
        fb.innerHTML = '\u26a0\ufe0f No answer selected. ' + explanations[q];
      }
    }
    document.getElementById('score').innerHTML = `Score: ${score} / ${Object.keys(answers).length}`;
  }
</script>
""")
